# Session 15: Text Mining with Vector Databases
This notebook demonstrates the application of text mining concepts using vector databases with `chromadb` and `sentence-transformers`.

Autor: César Diego Ruelas Flores

In [1]:
import sys
import os
import logging
import logging.config
import polars as pl

# Añadir el directorio src al PATH para importar módulos (PRIMERO SE DEBEN TENER LOS UTILS LISTOS)
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../src')))
from utils import (
    initialize_vector_db,
    create_or_get_collection,
    load_embedding_model,
    generate_embeddings,
    add_documents_to_collection,
    query_vector_db,
    get_collection_data
)

## VARIABLES GLOBALES

In [2]:
# Ruta para almacenar la base de datos vectorial ChromaDB
CHROMA_DB_PATH = "./chroma_db_tecsup"

# Nombre de la colección (tabla) en ChromaDB para información personal
COLLECTION_NAME_TECSUP = "Tecsup"

# Nombre del modelo de embedding por defecto (primer motor)
DEFAULT_EMBEDDING_MODEL = 'all-MiniLM-L6-v2'

# Nombre de un modelo de embedding alternativo para comparación (segundo motor)
ALTERNATIVE_EMBEDDING_MODEL = 'sentence-transformers/multi-qa-MiniLM-L6-cos-v1'

# Configuración del logging
LOGGING_CONFIG_PATH = os.path.abspath(os.path.join(os.getcwd(), '../configs/logging.conf'))
if os.path.exists(LOGGING_CONFIG_PATH):
    logging.config.fileConfig(LOGGING_CONFIG_PATH)
else:
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')

logger = logging.getLogger(__name__)

## Funciones (../src/utils.py)

In [3]:
%%writefile ../src/utils.py
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
import polars as pl
import logging
import os

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def initialize_vector_db(db_path: str = "./chroma_db") -> chromadb.Client:
    """
    Initializes a local ChromaDB client.

    Args:
        db_path (str): The path where the ChromaDB should be stored.

    Returns:
        chromadb.Client: An initialized ChromaDB client.
    """
    try:
        # CORRECCIÓN: Pasa la configuración para permitir el reinicio
        client = chromadb.PersistentClient(
            path=db_path,
            settings=Settings(allow_reset=True)
        )
        logger.info(f"ChromaDB client initialized at {db_path}")
        return client
    except Exception as e:
        logger.error(f"Error initializing ChromaDB client: {e}")
        raise

def create_or_get_collection(client: chromadb.Client, collection_name: str) -> chromadb.Collection:
    """
    Creates a new collection or gets an existing one in ChromaDB.

    Args:
        client (chromadb.Client): The ChromaDB client.
        collection_name (str): The name of the collection.

    Returns:
        chromadb.Collection: The ChromaDB collection object.
    """
    try:
        collection = client.get_or_create_collection(name=collection_name)
        logger.info(f"Collection '{collection_name}' created or retrieved successfully.")
        return collection
    except Exception as e:
        logger.error(f"Error creating or getting collection '{collection_name}': {e}")
        raise

def load_embedding_model(model_name: str = 'all-MiniLM-L6-v2') -> SentenceTransformer:
    """
    Loads a SentenceTransformer embedding model.

    Args:
        model_name (str): The name of the pre-trained model to load.

    Returns:
        SentenceTransformer: The loaded embedding model.
    """
    try:
        model = SentenceTransformer(model_name)
        logger.info(f"Embedding model '{model_name}' loaded successfully.")
        return model
    except Exception as e:
        logger.error(f"Error loading embedding model '{model_name}': {e}")
        raise

def generate_embeddings(model: SentenceTransformer, texts: list[str]) -> list[list[float]]:
    """
    Generates embeddings for a list of texts using the given model.

    Args:
        model (SentenceTransformer): The embedding model.
        texts (list[str]): A list of texts to embed.

    Returns:
        list[list[float]]: A list of embeddings, where each embedding is a list of floats.
    """
    try:
        embeddings = model.encode(texts).tolist()
        logger.info(f"Generated embeddings for {len(texts)} documents.")
        return embeddings
    except Exception as e:
        logger.error(f"Error generating embeddings: {e}")
        raise

def add_documents_to_collection(
    collection: chromadb.Collection,
    documents: list[str],
    embeddings: list[list[float]],
    ids: list[str]
) -> None:
    """
    Adds documents, their embeddings, and IDs to a ChromaDB collection.

    Args:
        collection (chromadb.Collection): The ChromaDB collection.
        documents (list[str]): The list of documents (texts).
        embeddings (list[list[float]]): The corresponding embeddings for the documents.
        ids (list[str]): Unique IDs for each document.
    """
    if not documents:
        logger.warning("Attempted to add an empty list of documents. Skipping.")
        return  # <-- Termina la ejecución aquí

    try:
        collection.add(
            documents=documents,
            embeddings=embeddings,
            ids=ids
        )
        logger.info(f"Added {len(documents)} documents to the collection.")
    except Exception as e:
        logger.error(f"Error adding documents to collection: {e}")
        # En un entorno de prueba, es mejor dejar que el error ocurra para detectarlo.
        # En producción, podrías manejarlo de otra forma.
        raise

def query_vector_db(
    collection: chromadb.Collection,
    query_embedding: list[list[float]],
    n_results: int = 2
) -> pl.DataFrame:
    """
    Queries the vector database for the most similar documents.

    Args:
        collection (chromadb.Collection): The ChromaDB collection to query.
        query_embedding (list[list[float]]): The embedding of the query text.
        n_results (int): The number of nearest results to retrieve.

    Returns:
        pl.DataFrame: A Polars DataFrame containing the query results (documents, distances, ids).
    """
    try:
        results = collection.query(
            query_embeddings=query_embedding,
            n_results=n_results,
            include=['documents', 'distances', 'metadatas']
        )
        # Convert results to Polars DataFrame
        if results and 'documents' in results and results['documents']:
            # Flatten the list of lists for documents, distances, and ids
            flat_documents = [doc for sublist in results['documents'] for doc in sublist]
            flat_distances = [dist for sublist in results['distances'] for dist in sublist]
            flat_ids = [idx for sublist in results['ids'] for idx in sublist]
            
            df = pl.DataFrame({
                "id": flat_ids,
                "document": flat_documents,
                "distance": flat_distances
            })
            logger.info(f"Query returned {len(flat_documents)} results.")
            return df
        else:
            logger.warning("No documents found for the given query.")
            return pl.DataFrame({"id": pl.Series([], dtype=pl.Utf8), "document": pl.Series([], dtype=pl.Utf8), "distance": pl.Series([], dtype=pl.Float64)})
    except Exception as e:
        logger.error(f"Error querying vector database: {e}")
        raise

def get_collection_data(collection: chromadb.Collection) -> pl.DataFrame:
    """
    Retrieves all documents, their IDs, and embeddings from a ChromaDB collection.

    Args:
        collection (chromadb.Collection): The ChromaDB collection.

    Returns:
        pl.DataFrame: A Polars DataFrame containing the IDs, documents, and embeddings.
    """
    try:
        data = collection.get(include=["embeddings", "documents"])
        if data and 'documents' in data and data['documents']:
            df = pl.DataFrame({
                "id": data['ids'],
                "document": data['documents'],
                "embedding": data['embeddings']
            })
            logger.info(f"Retrieved {len(data['documents'])} documents from collection.")
            return df
        else:
            logger.warning("No documents found in the collection.")
            return pl.DataFrame({"id": pl.Series([], dtype=pl.Utf8), "document": pl.Series([], dtype=pl.Utf8), "embedding": pl.Series([], dtype=pl.Object)})
    except Exception as e:
        logger.error(f"Error retrieving data from collection: {e}")
        raise

Overwriting ../src/utils.py


## Implementación Principal
Esta sección contiene la lógica principal del programa, dividida en partes para una mejor organización y demostración.

### Parte A: Creación y Carga de la Base de Datos Vectorial 'Tecsup'

En esta parte, se inicializará una base de datos vectorial local, se creará una colección llamada "Tecsup", y se cargará información personal del usuario y de compañeros, junto con sus embeddings generados por el modelo `all-MiniLM-L6-v2`.

In [4]:
logger.info("\n--- Parte A: Creación y Carga de la Base de Datos Vectorial 'Tecsup' ---")

2025-06-28 01:33:09,066 - __main__ - INFO - 
--- Parte A: Creación y Carga de la Base de Datos Vectorial 'Tecsup' ---


In [5]:
# 1. Inicializar el cliente ChromaDB
client = initialize_vector_db(CHROMA_DB_PATH)

2025-06-28 01:33:09,191 - chromadb.telemetry.product.posthog - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2025-06-28 01:33:09,316 - chromadb.telemetry.product.posthog - ERROR - Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [6]:
# 2. Crear o obtener la colección 'Tecsup'
collection_tecsup_default = create_or_get_collection(client, COLLECTION_NAME_TECSUP)

2025-06-28 01:33:09,341 - chromadb.telemetry.product.posthog - ERROR - Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [7]:
# 3. Cargar el modelo de embedding por defecto
model_default = load_embedding_model(DEFAULT_EMBEDDING_MODEL)

In [8]:
# 4. Definir la información personal y de compañeros
personal_data = [
    "Mi nombre es Jean Pierre Ruelas, soy estudiante de Ciencia de Datos y me especializo en Machine Learning.",
    "Mi compañero Juan Pérez está interesado en el procesamiento de lenguaje natural y trabaja con grandes modelos de lenguaje.",
    "Mi compañera Ana García es experta en visualización de datos y utiliza herramientas como Power BI y Tableau.",
    "Mi nombre es Alex Rodriguez, me gusta la inteligencia artificial y soy desarrollador de software.",
    "Mi compañera Sofía Martínez se enfoca en la ingeniería de datos y la construcción de pipelines ETL."
]

personal_data_ids = [f"info_{i+1}" for i in range(len(personal_data))]

In [9]:
# 5. Generar embeddings para la información personal
embeddings_default = generate_embeddings(model_default, personal_data)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [10]:
# 6. Añadir documentos a la colección 'Tecsup'
add_documents_to_collection(collection_tecsup_default, personal_data, embeddings_default, personal_data_ids)

logger.info("Información personal cargada en la base de datos 'Tecsup' con el modelo por defecto.")

2025-06-28 01:33:12,892 - chromadb.telemetry.product.posthog - ERROR - Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given
2025-06-28 01:33:12,974 - __main__ - INFO - Información personal cargada en la base de datos 'Tecsup' con el modelo por defecto.


In [11]:
# Opcional: Visualizar los datos cargados para verificar
logger.info("\nDatos cargados en la colección 'Tecsup' (modelo por defecto):")
df_tecsup_default = get_collection_data(collection_tecsup_default)
if not df_tecsup_default.is_empty():
    logger.info(df_tecsup_default.select(["id", "document"]))
else:
    logger.info("No hay datos en la colección.")

2025-06-28 01:33:12,997 - __main__ - INFO - 
Datos cargados en la colección 'Tecsup' (modelo por defecto):
2025-06-28 01:33:12,999 - chromadb.telemetry.product.posthog - ERROR - Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
2025-06-28 01:33:13,013 - __main__ - INFO - shape: (5, 2)
┌────────┬─────────────────────────────────┐
│ id     ┆ document                        │
│ ---    ┆ ---                             │
│ str    ┆ str                             │
╞════════╪═════════════════════════════════╡
│ info_1 ┆ Mi nombre es Jean Pierre Ruela… │
│ info_2 ┆ Mi compañero Juan Pérez está i… │
│ info_3 ┆ Mi compañera Ana García es exp… │
│ info_4 ┆ Mi nombre es Alex Rodriguez, m… │
│ info_5 ┆ Mi compañera Sofía Martínez se… │
└────────┴─────────────────────────────────┘


### Parte B: Realización de Consultas con el Primer Motor de Embedding

Aquí se creará un prompt y se realizará una consulta a la base de datos "Tecsup" utilizando el primer motor de embedding (`all-MiniLM-L6-v2`) para capturar respuestas relevantes.

In [12]:
logger.info("\n--- Parte B: Realización de Consultas con el Primer Motor de Embedding ---")

2025-06-28 01:33:13,044 - __main__ - INFO - 
--- Parte B: Realización de Consultas con el Primer Motor de Embedding ---


In [13]:
# 1. Definir un prompt de consulta
query_prompt_default = "¿Quién estudia ciencia de datos y le gusta el machine learning?"
logger.info(f"Prompt de consulta (modelo por defecto): '{query_prompt_default}'")

2025-06-28 01:33:13,086 - __main__ - INFO - Prompt de consulta (modelo por defecto): '¿Quién estudia ciencia de datos y le gusta el machine learning?'


In [14]:
# 2. Generar el embedding para la consulta utilizando el modelo por defecto
query_embedding_default = generate_embeddings(model_default, [query_prompt_default])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [15]:
# 3. Realizar la consulta a la base de datos vectorial
results_default_model = query_vector_db(collection_tecsup_default, query_embedding_default, n_results=2)

logger.info("\nResultados de la consulta (modelo por defecto):")
if not results_default_model.is_empty():
    logger.info(results_default_model)
else:
    logger.info("No se encontraron resultados para la consulta.")

2025-06-28 01:33:13,182 - chromadb.telemetry.product.posthog - ERROR - Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
2025-06-28 01:33:13,187 - __main__ - INFO - 
Resultados de la consulta (modelo por defecto):
2025-06-28 01:33:13,188 - __main__ - INFO - shape: (2, 3)
┌────────┬─────────────────────────────────┬──────────┐
│ id     ┆ document                        ┆ distance │
│ ---    ┆ ---                             ┆ ---      │
│ str    ┆ str                             ┆ f64      │
╞════════╪═════════════════════════════════╪══════════╡
│ info_1 ┆ Mi nombre es Jean Pierre Ruela… ┆ 0.483742 │
│ info_4 ┆ Mi nombre es Alex Rodriguez, m… ┆ 0.782105 │
└────────┴─────────────────────────────────┴──────────┘


### Parte C: Comparación con un Segundo Motor de Embedding

En esta sección, se utilizará un motor de embedding diferente (`multi-qa-MiniLM-L6-cos-v1`), se creará una nueva colección (o se usará la misma si se desea sobrescribir), se cargará la misma información y se realizará la misma consulta para comparar las respuestas obtenidas y observar cómo influye el modelo de embedding en los resultados.

In [16]:
logger.info("\n--- Parte C: Comparación con un Segundo Motor de Embedding ---")

2025-06-28 01:33:13,225 - __main__ - INFO - 
--- Parte C: Comparación con un Segundo Motor de Embedding ---


NOTA: Para una comparación justa, se debería usar una nueva colección o limpiar la existente 
para evitar la mezcla de embeddings de diferentes modelos en la misma colección. 
Para este ejemplo, crearemos una nueva colección para el segundo modelo.

In [17]:
COLLECTION_NAME_TECSUP_ALT = "Tecsup_AltEmbedding"

In [18]:
# 1. Cargar el modelo de embedding alternativo
model_alternative = load_embedding_model(ALTERNATIVE_EMBEDDING_MODEL)

In [19]:
# 2. Crear o obtener una nueva colección para el modelo alternativo
collection_tecsup_alt = create_or_get_collection(client, COLLECTION_NAME_TECSUP_ALT)

2025-06-28 01:33:14,841 - chromadb.telemetry.product.posthog - ERROR - Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [20]:
# 3. Generar embeddings para la misma información personal con el modelo alternativo
embeddings_alternative = generate_embeddings(model_alternative, personal_data)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [21]:
# 4. Añadir documentos a la nueva colección
add_documents_to_collection(collection_tecsup_alt, personal_data, embeddings_alternative, personal_data_ids)

logger.info("Información personal cargada en la base de datos 'Tecsup_AltEmbedding' con el modelo alternativo.")

2025-06-28 01:33:15,022 - chromadb.telemetry.product.posthog - ERROR - Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given
2025-06-28 01:33:15,094 - __main__ - INFO - Información personal cargada en la base de datos 'Tecsup_AltEmbedding' con el modelo alternativo.


In [22]:
# 5. Definir el mismo prompt de consulta
query_prompt_alternative = "¿Quién estudia ciencia de datos y le gusta el machine learning?"
logger.info(f"Prompt de consulta (modelo alternativo): '{query_prompt_alternative}'")

2025-06-28 01:33:15,112 - __main__ - INFO - Prompt de consulta (modelo alternativo): '¿Quién estudia ciencia de datos y le gusta el machine learning?'


In [23]:
# 6. Generar el embedding para la consulta utilizando el modelo alternativo
query_embedding_alternative = generate_embeddings(model_alternative, [query_prompt_alternative])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [24]:
# 7. Realizar la consulta a la base de datos vectorial con el modelo alternativo
results_alternative_model = query_vector_db(collection_tecsup_alt, query_embedding_alternative, n_results=2)

logger.info("\nResultados de la consulta (modelo alternativo):")
logger.info("\n--- Comparación de Resultados ---")

# Usa f-strings para formatear la salida y evita el operador '+'
if not results_default_model.is_empty():
    logger.info(f"Modelo Default (all-MiniLM-L6-v2) Resultados:\n{results_default_model}")
else:
    logger.info("No hay resultados para el modelo Default")

# Haz lo mismo para el segundo modelo y recuerda quitar .to_string()
if not results_alternative_model.is_empty():
    logger.info(f"\nModelo Alternativo (multi-qa-MiniLM-L6-cos-v1) Resultados:\n{results_alternative_model}")
else:
    logger.info("No hay resultados para el modelo Alternativo")

logger.info("\nSe puede observar cómo los diferentes modelos de embedding pueden influir en la relevancia y el orden de los resultados de la consulta.")


2025-06-28 01:33:15,233 - chromadb.telemetry.product.posthog - ERROR - Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
2025-06-28 01:33:15,241 - __main__ - INFO - 
Resultados de la consulta (modelo alternativo):
2025-06-28 01:33:15,243 - __main__ - INFO - 
--- Comparación de Resultados ---
2025-06-28 01:33:15,245 - __main__ - INFO - Modelo Default (all-MiniLM-L6-v2) Resultados:
shape: (2, 3)
┌────────┬─────────────────────────────────┬──────────┐
│ id     ┆ document                        ┆ distance │
│ ---    ┆ ---                             ┆ ---      │
│ str    ┆ str                             ┆ f64      │
╞════════╪═════════════════════════════════╪══════════╡
│ info_1 ┆ Mi nombre es Jean Pierre Ruela… ┆ 0.483742 │
│ info_4 ┆ Mi nombre es Alex Rodriguez, m… ┆ 0.782105 │
└────────┴─────────────────────────────────┴──────────┘
2025-06-28 01:33:15,246 - __main__ - INFO - 
Modelo Alternativo (multi-qa-MiniLM-L6-cos-v1) Resul

## Testing

Para asegurar la calidad y el correcto funcionamiento de las funciones definidas en `src/utils.py`, ejecutamos los tests unitarios con `pytest`.

In [25]:
%%writefile ../tests/test_utils.py
import pytest
import os
import chromadb
import shutil
import time 
import polars as pl
from sentence_transformers import SentenceTransformer
from chromadb.api.client import Client as ChromaClient
from chromadb.api.models.Collection import Collection

# Importar las funciones que vamos a probar desde el archivo utils
from src.utils import (
    initialize_vector_db,
    create_or_get_collection,
    load_embedding_model,
    generate_embeddings,
    add_documents_to_collection,
    query_vector_db,
    get_collection_data
)

# --- Fixtures para las pruebas (configuración que se reutiliza) ---

@pytest.fixture(scope="function")
def setup_teardown_db(monkeypatch):
    """
    Usa monkeypatch para reemplazar el cliente de disco por uno en memoria,
    manejando los argumentos de forma inteligente.
    """
    monkeypatch.setattr(
        chromadb,
        "PersistentClient",
        lambda path, settings: chromadb.EphemeralClient(settings=settings)
    )

    # Ahora esta llamada es segura. Nuestra lambda maneja los argumentos.
    client = initialize_vector_db("dummy_path_for_test")
    yield client

    # La limpieza en memoria es simple y ahora funcionará sin errores.
    client.reset()

@pytest.fixture(scope="module")
def embedding_model():
    """
    Carga el modelo de embedding una sola vez para todas las pruebas.
    """
    return load_embedding_model('all-MiniLM-L6-v2')

# --- Casos de Prueba ---

def test_initialize_vector_db(setup_teardown_db):
    """
    Prueba que el cliente de la base de datos se inicializa correctamente.
    """
    client = setup_teardown_db
    assert isinstance(client, ChromaClient)
    assert os.path.exists("./test_chroma_db")

def test_create_or_get_collection(setup_teardown_db):
    """
    Prueba que se puede crear y obtener una colección.
    """
    client = setup_teardown_db
    collection_name = "test_collection"
    collection = create_or_get_collection(client, collection_name)
    assert isinstance(collection, Collection)
    assert collection.name == collection_name

def test_load_embedding_model():
    """
    Prueba que el modelo de embedding se carga correctamente.
    """
    model = load_embedding_model('all-MiniLM-L6-v2')
    assert isinstance(model, SentenceTransformer)

def test_generate_embeddings(embedding_model):
    """
    Prueba que se generan embeddings correctamente.
    """
    texts = ["hello world", "test sentence"]
    embeddings = generate_embeddings(embedding_model, texts)
    assert isinstance(embeddings, list)
    assert len(embeddings) == len(texts)

def test_query_vector_db(setup_teardown_db, embedding_model):
    """
    Prueba que las consultas a la base de datos devuelven resultados.
    """
    client = setup_teardown_db
    collection = create_or_get_collection(client, "query_test_collection")

    docs = ["apple is a fruit", "banana is a fruit", "car is a vehicle"]
    ids = ["d1", "d2", "d3"]
    embeddings = generate_embeddings(embedding_model, docs)
    add_documents_to_collection(collection, docs, embeddings, ids)

    query_embedding = generate_embeddings(embedding_model, ["What is a fruit?"])
    results_df = query_vector_db(collection, query_embedding, n_results=2)

    assert isinstance(results_df, pl.DataFrame)
    assert len(results_df) == 2

def test_get_collection_data(setup_teardown_db, embedding_model):
    """
    Prueba que se pueden obtener todos los datos de una colección.
    """
    client = setup_teardown_db
    collection = create_or_get_collection(client, "get_data_test_collection")

    docs = ["item A", "item B"]
    ids = ["id_A", "id_B"]
    embeddings = generate_embeddings(embedding_model, docs)
    add_documents_to_collection(collection, docs, embeddings, ids)

    df = get_collection_data(collection)
    assert isinstance(df, pl.DataFrame)
    assert len(df) == 2
    assert "item A" in df["document"].to_list()

def test_query_vector_db_no_results(setup_teardown_db, embedding_model):
    """
    Prueba que una consulta sin resultados devuelve un DataFrame vacío.
    """
    client = setup_teardown_db
    collection = create_or_get_collection(client, "empty_query_collection")
    query_embedding = generate_embeddings(embedding_model, ["non-existent item"])
    results_df = query_vector_db(collection, query_embedding, n_results=2)
    assert results_df.is_empty()

def test_add_documents_empty_lists(setup_teardown_db):
    """
    Prueba que al intentar añadir listas vacías, la operación no hace nada y no falla.
    """
    client = setup_teardown_db
    collection = create_or_get_collection(client, "empty_add_collection")
    
    # Esta llamada ahora no debería hacer nada gracias a la guarda en utils.py
    add_documents_to_collection(collection, [], [], [])
    
    df = get_collection_data(collection)
    assert df.is_empty()

Overwriting ../tests/test_utils.py


In [26]:
!pytest ../tests/test_utils.py -v

============================= test session starts =============================
platform win32 -- Python 3.13.5, pytest-8.4.1, pluggy-1.6.0 -- c:\Users\AzShet\Documents\Jupyter_LAB\jupyter_projects\5to_ciclo\DataMining\lab15\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\AzShet\Documents\Jupyter_LAB\jupyter_projects\5to_ciclo\DataMining\lab15\Data_Mining-LAB15
plugins: anyio-4.9.0
collecting ... collected 8 items

..\tests\test_utils.py::test_initialize_vector_db PASSED                 [ 12%]
..\tests\test_utils.py::test_create_or_get_collection PASSED             [ 25%]
..\tests\test_utils.py::test_load_embedding_model PASSED                 [ 37%]
..\tests\test_utils.py::test_generate_embeddings PASSED                  [ 50%]
..\tests\test_utils.py::test_query_vector_db PASSED                      [ 62%]
..\tests\test_utils.py::test_get_collection_data PASSED                  [ 75%]
..\tests\test_utils.py::test_query_vector_db_no_results PASSED           [ 87%]
..